In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
files[0]

RawRepositoryFile(filename='01-agentic-rag/lessons/01-intro.md', content='# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are 

In [3]:
from ingest_course_lessons import lesson_data, build_index

documents = lesson_data()

len(documents)

72

In [4]:

index = build_index(documents)

question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [5]:
from openai import OpenAI
from rag_helper_courses import RAGBase
from dotenv import load_dotenv

openai_client = OpenAI()

rag = RAGBase(
    index=index,
    llm_client=openai_client
)

answer, usage = rag.rag(question)

print(usage.input_tokens)


7136


In [6]:
from gitsource import chunk_documents

In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))

295


In [8]:
chunk_index = build_index(chunks)

chunk_assistant = RAGBase(
    index=chunk_index,
    llm_client=openai_client,
)

chunk_answer, chunk_usage = chunk_assistant.rag(question)

print(chunk_answer)
print(chunk_usage.input_tokens)

The loop keeps calling the model in a `while True` loop and checks each response for `function_call` items. If it finds any, it runs the tool, appends the tool result to the message history, and keeps going. It stops when a turn has no function calls:

```python
if has_function_calls == False:
    break
```

So the stopping condition is: **no function calls this turn means the agent is done.**
2319


In [9]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the course lesson chunks for relevant information.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course lessons."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [10]:
import json


search_call_count = 0


def search(query):
    """
    Search the course lesson chunks for relevant information.
    """
    global search_call_count

    search_call_count += 1

    return chunk_index.search(
        query,
        num_results=5
    )

In [11]:
agent_instructions = """
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
""".strip()

In [17]:
import json

counter = {"search_count": 0}


def search(query):
    counter["search_count"] += 1

    return chunk_index.search(
        query,
        num_results=5
    )


search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the course lesson chunks.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string"
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


def run_agent(question):
    messages = [
        {
            "role": "developer",
            "content": """
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
""".strip()
        },
        {
            "role": "user",
            "content": question
        }
    ]

    while True:
        response = openai_client.responses.create(
            model="gpt-5.4-mini",
            input=messages,
            tools=[search_tool],
        )

        messages.extend(response.output)

        tool_was_called = False
        answer = None

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)

                args = json.loads(item.arguments)
                result = search(args["query"])

                messages.append({
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": json.dumps(result)
                })

                tool_was_called = True

            elif item.type == "message":
                answer = item.content[0].text
                print("ASSISTANT:")
                print(answer)

        if not tool_was_called:
            return answer


counter["search_count"] = 0

answer = run_agent(
    "How does the agentic loop work, and how is it different from plain RAG?"
)



function_call: search {"query":"agentic loop RAG lesson"}
function_call: search {"query":"plain RAG agentic loop"}
function_call: search {"query":"agentic loop retrieval generation iteration"}
function_call: search {"query":"RAG vs agentic loop"}
ASSISTANT:
The agentic loop is basically a repeated **LLM → tool call → tool result → LLM** cycle.

From the lesson, the core pattern is:

1. Send the user question plus any instructions to the model.
2. The model may return one or more **function/tool calls**.
3. Your code executes those tools and appends the results back into the message history.
4. Repeat.
5. Stop when the model returns a normal answer **with no more tool calls**.

So the LLM is not just answering once — it is **deciding what to do next at each step**. The lesson describes this as a `while` loop that keeps going until there are no function calls left.

### How that differs from plain RAG

Plain RAG is a **fixed pipeline**:

```python
def rag(question):
    search_results = 

In [18]:
counter["search_count"] = 0

answer = run_agent(
    "How does the agentic loop work, and how is it different from plain RAG?"
)

print("Search calls:", counter["search_count"])

function_call: search {"query":"agentic loop RAG difference course lesson"}
function_call: search {"query":"plain RAG agentic loop retrieval generation iteration"}
function_call: search {"query":"agentic loop steps retrieve reason act reflect"}
function_call: search {"query":"RAG vs agentic loop lesson chunks"}
ASSISTANT:
The **agentic loop** is basically a repeating cycle:

1. Send the user question plus instructions to the LLM
2. The LLM decides whether to call a tool
3. Your code executes the tool
4. You send the tool result back to the LLM
5. Repeat until the LLM returns a final answer with **no more tool calls**

So the model is “in the driver’s seat,” and your code just keeps the loop going. The lesson describes it as a `while` loop that keeps calling the model, running any function calls it returns, and stopping when the model is done.

### How it differs from plain RAG

**Plain RAG** is simpler and fixed:

```python
search_results = search(question)
prompt = build_prompt(questi